# Bonus de Clasificación
## Análisis discriminante y comparación de modelos

**Objetivo:** Utilizar análisis de discriminante lineal (LDA) y cuádratico (QDA); Naive Bayes y comparar con los modelos de la tutorial de Logit. También, haremos el análisis de performance con la curva ROC para comparar entre métodos.

Veremos:
- Clasificación
- Análisis de discriminante lineal (LDA) y cuadrático (QDA)
- Comparación de modelos: logit, LDA, QDA, Naive Bayes


In [ ]:
import os  
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt  
import seaborn as sns
import statsmodels.api as sm 
from ISLP import load_data

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# metricas de desempeño de clasificiacion
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, recall_score 
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from sklearn.metrics import RocCurveDisplay
#from sklearn.metrics import plot_roc_curve

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA
from sklearn.neighbors import KNeighborsClassifier




## Comparación de modelos usando datos del mercado de acciones 

En este ejemplo, vamos a usar datos del [Stock Market S&P](https://islp.readthedocs.io/en/latest/datasets/Weekly.html) libro ISLP. 
Esta base contiene los retornos porcentuales del S&P 500 stock index por 1250 días, desde inicios de 2001 hasta el final de 2005. Para cada fecha, tenemos:
- Lag1, Lag2,..., Lag5: retornos porcentuales de cada uno de los días anteriores.
- Volume: volumen de acciones negociadas (número de acciones diarias negociadas en miles de millones de dólares)
- Today: retorno porcentual de hoy
- Direction: variable binaria que toma valores "Down" y "Up" indicando si el mercado tuvo un retorno positivo o negativo.


In [ ]:
# Cargamos los datos de Smarket.
smarket = load_data('Smarket')
smarket

In [ ]:
smarket.corr(numeric_only=True).round(2) # con la opcion numeric_only=True hacemo que no tenga en cuenta Direction (string)

In [ ]:
colormap = plt.cm.viridis
plt.figure(figsize=(8,8))
plt.title('Correlacion de Pearson entre las Xs', y=1.05, size=15)
sns.heatmap(smarket.iloc[:,:-1].corr(),
            linewidths=0.1,
            vmax=1.0, 
            square=True, 
            cmap=colormap, 
            linecolor='white', 
            annot=True)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4)) # Un tamaño de gráfico más grande suele verse mejor
smarket.plot(y='Volume', linewidth=.5, ax=ax)
ax.set_title('Tendencias del Volumen del Índice S&P 500', fontsize=16)
ax.set_ylabel('Volumen (promedio diario de\nacciones vendidas en billones)', fontsize=12)
ax.set_xlabel('Orden de Fecha', fontsize=12)
plt.tight_layout() # Ajusta automáticamente los márgenes
plt.show()

In [ ]:
print(smarket['Direction'].value_counts().round(2))

In [ ]:
print(smarket.groupby('Direction').mean().round(2))

Vamos a usar los distintos modelos de clasificación para predicir si sube el indice S&P (**'Direction=Up'**) usando los rezagos (lags) 1 a 5 y el Volumen como predictores.

#### Consideración temporal para el entrenamiento y testeo
En los casos en los que nuestra base de datos tienen una dimensión temporal, el tiempo es una variable "ordenadora" de los datos. Por lo tanto, lo logico es *entrenar* nuestros modelos para la selección de complejidad con **algunos años**, y *testear* nuestro "mejor" modelo en los **últimos año**. De nuevo, con una logica similar a antes, queremos usar aproximadamente un 70% 0 80% de los datos para entrenar y un 30% o 20% para testear.

In [ ]:
# Hacemos el split de la base entre train y test:
train = smarket[smarket.Year < 2005]
test = smarket[smarket.Year >= 2005]
    
ytrain = train['Direction']
ytrain = ytrain.replace('Up', 1)
ytrain = ytrain.replace('Down', 0)
print('\n Var dependiente de entrenamiento', ytrain.shape)

Xtrain = train[['Lag1', 'Lag2', 'Lag3', 'Lag4', 'Lag5', 'Volume']]
Xtrain = sm.add_constant(Xtrain)
print('\n X de entrenamiento', Xtrain.shape)

ytest = test['Direction']
ytest = ytest.replace('Up', 1)
ytest = ytest.replace('Down', 0)
print('\n Var dependiente de testeo',ytest.shape)

Xtest = test[['Lag1', 'Lag2', 'Lag3', 'Lag4', 'Lag5', 'Volume']]
Xtest = sm.add_constant(Xtest)
print('\n X de testeo', Xtest.shape)

In [ ]:
# Chequeamos entonces las obsservaciones a predecir en cada base
print(ytest.value_counts().round(2))

## Análisis Discriminante Lineal (LDA)

[LinearDiscriminantAnalysis()](http://scikit-learn.org/stable/modules/generated/sklearn.discriminant_analysis.LinearDiscriminantAnalysis.html): Es un clasificador que utiliza un límite lineal para distinguir las categorías, generado a través del ajuste de densidades condicionales de las clases y utilizando la regla de Bayes.

El modelo ajusta una densidad gaussiana a cada clase, asumiendo que todas las clases comparten la misma matriz de covarianza.

In [ ]:
lda = LinearDiscriminantAnalysis() # usamos la configuración de default
lda_results = lda.fit(Xtrain, ytrain)

Podemos ver algunas partes de la formula en LDA, el vector de medias por clase de Y

In [ ]:
# Chequeamos las "clases"/categorias de nuestra variable dependiente UP
lda_results.classes_

In [ ]:
lda_results.priors_.round(2) # proporciones de Y de cada clase, prob(Y=k) prior

##### Efectos Marginales de un predictor X en la probabilidad $Pr(Y=1|X)$
Podemos visualizar cómo cambia la probabilidad de una clase de nuestra variable dependiente en funcion de los valores de nuestros predictores.

In [ ]:
# Seleccionamos `Volume`
x = Xtrain['Volume']

# Creamos un rango de valores para el predictor X
x_range = np.linspace(x.min(), x.max(), 1000)

In [ ]:
# Creamos un array con los valores medios de las demás variables
x_mean = Xtrain.mean().values

# Creamos un array con la misma forma que Xtrain pero con x_range en la columna 'Volume'
x_range_array = np.tile(x_mean, (len(x_range), 1))
x_range_array[:, Xtrain.columns == 'Volume'] = x_range[:, np.newaxis]

# Calcula las probabilidades predichas para el rango de valores de 'Volume'
y_prob_lda = lda_results.predict_proba(x_range_array)[:, 1]

# Calcula la probabilidad acumulada
y_prob_cum = np.cumsum(y_prob_lda) / np.sum(y_prob_lda)

In [ ]:
# Grafica las probabilidades predichas en función de X
plt.plot(x_range, y_prob_cum)
plt.xlabel('Volume')
plt.ylabel('Probabilidad de Y= Up')
plt.title('Efecto marginal del Volumen en\nla probabilidad de que suba el índice')
plt.show()

Predecimos con el modelo de análisis discriminante lineal usando las X test.

In [ ]:
y_pred_lda = lda_results.predict(Xtest)
# Consejo: siempre mirar los objetos que uno va creando
y_pred_lda

### Repaso: Medidas de precisión 

Dependiendo la prioridad del problema seguramente vamos a querer usar diferentes métricas. Scikit learn tiene muchas métricas que pueden explorar en el módulo [metrics](https://scikit-learn.org/stable/modules/model_evaluation.html)

- Sensitivity o Recall o True Positive Rate: TP rate = TP/P
- Specificity o True Negative Rate: 1 - FP rate = TN/N
- False Positive Rate o False Alarm Rate: FP rate = FP/N
- False Negative Rate: FN rate = FN/P
- Precision o Positive Predicted Value: TP/(TP+FP)
- Accuracy: (TP+TN)/(P+N)

Nota: Cuidado con las traducciones! "Accuracy" lo pueden encontrar traducido como "precisión" y eso puede generar confusión con la medida "precision" (o positive predicted value). Mi sugerencia es traducir "accuracy" como "exactitud".


[Matriz de confusión](https://www.unite.ai/what-is-a-confusion-matrix/)
<center>
<img src="https://www.unite.ai/wp-content/uploads/2019/12/Preventive_Medicine-e1576294312614.png" width="1000">

</center>

In [ ]:
accuracy_lda = accuracy_score(ytest, y_pred_lda)
print("La accuracy del modelo es: %.2f" %accuracy_lda)

#### Repaso: Matriz de Confusión
La matriz de confusión de sklearn pone en las filas las Y reales y las columnas las Y predichas. Muestra así los valores:

                               predicción
                         real   tn fp
                                fn tp

In [ ]:
confusion_matrix(ytest,y_pred_lda)

### Curva ROC                  
ROC: Receiver Operating Characteristics

Veremos como utilizar las funciones:

-  [roc_curve](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_curve.html#sklearn.metrics.roc_curve): computa la curva de ROC
- [roc_auc_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html#sklearn.metrics.roc_auc_score): Computa el area bajo la curva de ROC de los scores predichos.
- [RocCurveDisplay](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.RocCurveDisplay.html#sklearn.metrics.RocCurveDisplay): Sirve para visualizar la curva de ROC. Con el mismo fin existe [plot_roc_curve](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.plot_roc_curve.html#sklearn.metrics.plot_roc_curve)

Para estas funciones necesitamos predecir dicha probabilidad condicional $p=Pr(Y=1|X)$ o *score*

In [ ]:
# Volvemos a predecir la probabilidad del modelo afuera de la muestra
y_prob_lda=lda_results.predict_proba(Xtest)
y_prob_lda.shape

In [ ]:
y_prob_lda[:5,:]

In [ ]:
# Computamos la tasa de verdaderos positivos (tpr) y falsos positivos (tpr) para construir la curva ROC
fpr, tpr, thresholds = roc_curve(ytest, y_prob_lda[:,1]) 

In [ ]:
print('Thresholds:', thresholds)   # Estos son los umbrales c en las slides
print('FPR:', fpr)
print('TPR:', tpr)

In [ ]:
#Area under curve (es una medida de precisón)
auc_lda = roc_auc_score(ytest, y_prob_lda[:,1]) 
print('AUC: %.2f' % auc_lda)

In [ ]:
display = RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=auc_lda, estimator_name='LDA')
display.plot() 
plt.title('Curva ROC de LDA')
plt.plot([0, 1], [0, 1], color='red', linestyle='--')
plt.show() 

## Analisis Discriminante Cuadrático (QDA)
Usamos la funcion de Análisis Discriminante Cuadrático: [QuadraticDiscriminantAnalysis()](https://scikit-learn.org/stable/modules/generated/sklearn.discriminant_analysis.QuadraticDiscriminantAnalysis.html)

In [ ]:
# Estimamos con la base de entrenamiento
qda = QDA() # usamos la configuración de default
qda_results = qda.fit(Xtrain, ytrain)

# Predecimos con la base de testeo
y_qda = qda_results.predict(Xtest) 
y_qda

In [ ]:
# Probabilidades del modelo QDA afuera de la muestra
y_prob_qda=qda.predict_proba(Xtest)
y_prob_qda.shape

In [ ]:
y_prob_qda[:5,:]

Una alternativa para solucionar estos problemas de colinealidad es rescalar las variables.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Estandarizamos las variables
scaler = StandardScaler()
Xtrain_scaled = scaler.fit_transform(Xtrain)
Xtest_scaled = scaler.transform(Xtest)

In [ ]:
# Estima el modelo QDA con regularización
qda = QDA(reg_param=0.1)
qda_results = qda.fit(Xtrain_scaled, ytrain)

# Predecimos con la base de testeo
y_pred_qda = qda_results.predict(Xtest_scaled)
y_pred_qda

In [ ]:
# Probabilidades del modelo QDA afuera de la muestra
y_prob_qda=qda_results.predict_proba(Xtest)
y_prob_qda.shape

In [ ]:
y_prob_qda[:5,:].round(2)

##### Performance de QDA

In [ ]:
# Matriz de resultados
conf_mat_qda = confusion_matrix(ytest,y_pred_qda)
print(conf_mat_qda)  

# Precisión afuera de la muestra
accuracy_qda = accuracy_score(ytest, y_pred_qda)
print("La accuracy del modelo es: %.2f" %accuracy_qda)

In [ ]:
# Computamos la tasa de verdaderos positivos (tpr) y falsos positivos (tpr) para construir la curva ROC
fpr, tpr, thresholds = roc_curve(ytest, y_prob_qda[:,1]) 

#Area under curve (es una medida de precisón)
auc_qda = roc_auc_score(ytest, y_prob_qda[:,1]) 
print('AUC: %.2f' % auc_lda)

# Curva ROC
display = RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=auc_lda, estimator_name='QDA')
display.plot() 
plt.title('Curva ROC de QDA')
plt.plot([0, 1], [0, 1], color='red', linestyle='--')
plt.show() 

#### Comparemos con modelos anteriores

Vamos a usar el modelo de **regresión logística** para predicir 'Direction' usando los lags 1 a 5 y Volume. 

In [ ]:
# Regresión logística
logit_model = LogisticRegression(penalty=None).fit(Xtrain,ytrain)

# Probabilidades predichas
y_pred_logit = logit_model.predict(Xtest)
y_pred_logit

In [ ]:
# Probabilidades de logit
y_prob_log = logit_model.predict_proba(Xtest)

# Matriz de confusión
conf_mat = confusion_matrix(ytest, y_pred_logit) 

print('Confusion Matrix:\n', conf_mat) 
print('Accuracy Score:',accuracy_score(ytest, y_pred_logit)) # Cantidad de (vp+vn) sobre total
# Recordar: acá la matriz de confusión tiene en las filas los valores ciertos y en las columnas los valores predichos

In [ ]:
# Computamos la tasa de verdaderos positivos (tpr) y falsos positivos (tpr) para construir la curva ROC
fpr, tpr, thresholds = roc_curve(ytest, y_prob_log[:,1]) 

#Area under curve (es una medida de precisón)
auc = roc_auc_score(ytest, y_prob_log[:,1]) 
print('AUC: %.2f' % auc)

#Curva ROC
display = RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=auc, estimator_name='logit')
display.plot()  
plt.title('Curva ROC de Logit')
plt.plot([0, 1], [0, 1], color='red', linestyle='--')
plt.show() 

### Naive Bayes
También podemos hacer la predicción de $Pr(Y=k|X)$ y la regla de Bayes, levantando el supuesto de normalidad de $X|Y$, pero haciendo el supuesto de independencia de $X$. Implementamos la funcion [GaussianNB()](https://scikit-learn.org/dev/modules/generated/sklearn.naive_bayes.GaussianNB.html), para el modelo de clasificador de Naive Bayes, pero también podriamos estimar las densidades con metodo de Kernels.  



In [ ]:
from sklearn.naive_bayes import GaussianNB

In [ ]:
NB = GaussianNB() 
NB.fit(Xtrain, ytrain)

In [ ]:
y_pred_nb= NB.predict(Xtest) 
y_pred_nb

In [ ]:
# Matriz de resultados
conf_mat4 = confusion_matrix(ytest, y_pred_nb)
print(conf_mat4)  

In [ ]:
# Probabilidades de Naive Bayes
y_prob_nb = NB.predict_proba(Xtest)

# AUC y ROC
auc = roc_auc_score(ytest, y_prob_nb[:,1])
print('AUC Naive Bayes: %.2f' % auc)
fpr, tpr, thresholds = roc_curve(ytest, y_prob_nb[:,1])

display = RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=auc, estimator_name='Naive Bayes')
display.plot()  
plt.plot([0, 1], [0, 1], color='red', linestyle='--')
plt.show() 

#### Ejercicios para probar y hacer las comparaciones finales
1. Estimar KNN eligiendo $K$ (numero de vecinos) por cross-validation
2. Crea un grafico con todas las curvas ROC de los modelos
3. Estimar logit con penalidad L1 y L2